In [ ]:
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
# -----------------------------
# Paths and run configuration
# -----------------------------
GENES_ROOT = Path("/kaggle/input/datasets/mruhaib/100_genes")

WORK_ROOT = Path("/kaggle/working/100_gene_eval")
VCF_DIR = WORK_ROOT / "VCF"
ENC_DIR = WORK_ROOT / "Encodings"
RESULTS_DIR = WORK_ROOT / "Results"

# Remaining genes config:
# From 100 sorted FASTA files, first 77 were already generated.
ALREADY_GENERATED_COUNT = 77
EXPECTED_REMAINING = 23

# Keep these False so reruns only fill missing files for remaining genes.
REBUILD_VCF = False
REBUILD_ENCODINGS = False

for p in [WORK_ROOT, VCF_DIR, ENC_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"GENES_ROOT : {GENES_ROOT}")
print(f"WORK_ROOT  : {WORK_ROOT}")
print(f"VCF_DIR    : {VCF_DIR}")
print(f"ENC_DIR    : {ENC_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")
print(f"ALREADY_GENERATED_COUNT: {ALREADY_GENERATED_COUNT}")
print(f"EXPECTED_REMAINING     : {EXPECTED_REMAINING}")

In [ ]:
# -----------------------------
# FASTA -> VCF
# -----------------------------
def parse_fasta_records(path):
    records = []
    cur_id = None
    cur_seq = []
    with open(path, "r", encoding="utf-8") as fh:
        for raw in fh:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(">"):
                if cur_id is not None:
                    records.append((cur_id, "".join(cur_seq).upper()))
                cur_id = line[1:].split()[0]
                cur_seq = []
            else:
                cur_seq.append(line)
    if cur_id is not None:
        records.append((cur_id, "".join(cur_seq).upper()))
    return records

def align_query_to_ref(ref_seq, query_seq):
    """Return query nucleotides aligned to each reference position ('-' for deletion)."""
    out = ["-"] * len(ref_seq)
    sm = SequenceMatcher(None, ref_seq, query_seq, autojunk=False)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            for k in range(i2 - i1):
                out[i1 + k] = query_seq[j1 + k]
        elif tag == "replace":
            overlap = min(i2 - i1, j2 - j1)
            for k in range(overlap):
                out[i1 + k] = query_seq[j1 + k]
        elif tag == "delete":
            pass
        elif tag == "insert":
            pass
    return out

def write_gene_vcf(fasta_path, out_vcf_path):
    records = parse_fasta_records(fasta_path)
    if len(records) == 0:
        raise ValueError(f"No sequences in {fasta_path}")

    gene = fasta_path.stem
    strain_ids = [rid for rid, _ in records]
    ref_seq = records[0][1]
    n_strains = len(records)

    matrix = [[None] * n_strains for _ in range(len(ref_seq))]
    for i, b in enumerate(ref_seq):
        matrix[i][0] = b

    for s_idx, (_, qseq) in enumerate(records[1:], start=1):
        aligned = align_query_to_ref(ref_seq, qseq)
        for pos, nuc in enumerate(aligned):
            matrix[pos][s_idx] = nuc

    with open(out_vcf_path, "w", encoding="utf-8") as fh:
        fh.write("##fileformat=VCFv4.2\n")
        fh.write(f"##reference={gene}_strain0\n")
        fh.write(f"##CHROM={gene}\n")
        fh.write("##INFO=<ID=AC,Number=A,Type=Integer,Description=\"Allele count in called genotypes\">\n")
        fh.write("##INFO=<ID=AF,Number=A,Type=Float,Description=\"Allele frequency among called genotypes\">\n")
        fh.write("##INFO=<ID=NS,Number=1,Type=Integer,Description=\"Number of strains with a non-missing call\">\n")
        fh.write("##INFO=<ID=VAR_TYPE,Number=1,Type=String,Description=\"SNP | DEL | MIXED\">\n")
        fh.write("##FORMAT=<ID=GT,Number=1,Type=String,Description=\"Genotype (allele index; . = missing/deletion)\">\n")

        header = [
            "#CHROM",
            "POS",
            "ID",
            "REF",
            "ALT",
            "QUAL",
            "FILTER",
            "INFO",
            "FORMAT",
        ] + strain_ids
        fh.write("\t".join(header) + "\n")

        variant_sites = 0
        for pos in range(len(ref_seq)):
            col = matrix[pos]
            ref_nuc = col[0]
            if ref_nuc in (None, "-", "N"):
                continue

            calls = [x for x in col if x is not None]
            unique = set(calls) - {"-"}
            if len(unique) <= 1 and "-" not in calls:
                continue

            alts = sorted(unique - {ref_nuc})
            has_del = "-" in calls
            if has_del:
                alts.append("*")
            if not alts:
                continue

            allele_idx = {ref_nuc: "0"}
            for i, a in enumerate(alts, start=1):
                allele_idx[a] = str(i)

            gts = []
            for nuc in col:
                if nuc is None:
                    gts.append(".")
                elif nuc == "-":
                    gts.append(allele_idx.get("*", "."))
                else:
                    gts.append(allele_idx.get(nuc, "."))

            n_called = sum(1 for g in gts if g != ".")
            ac = [gts.count(str(i)) for i in range(1, len(alts) + 1)]
            af = [(a / n_called if n_called > 0 else 0.0) for a in ac]

            alt_display = [a for a in alts if a != "*"]
            if not alt_display:
                var_type = "DEL"
            elif all(len(a) == 1 and a.upper() in "ACGTN" for a in alt_display):
                var_type = "SNP" if not has_del else "MIXED"
            else:
                var_type = "MIXED"

            af_str = ",".join(f"{x:.6f}" for x in af)
            info = f"AC={','.join(map(str, ac))};AF={af_str};NS={n_called};VAR_TYPE={var_type}"
            row = [
                gene,
                str(pos + 1),
                ".",
                ref_nuc,
                ",".join(alts),
                ".",
                "PASS",
                info,
                "GT",
            ] + gts
            fh.write("\t".join(row) + "\n")
            variant_sites += 1

    return {"gene": gene, "n_strains": n_strains, "n_variant_sites": variant_sites}

def discover_gene_fastas(root):
    return sorted(root.rglob("*.fasta"))

all_gene_fastas = discover_gene_fastas(GENES_ROOT)
if len(all_gene_fastas) <= ALREADY_GENERATED_COUNT:
    raise ValueError(
        f"Discovered {len(all_gene_fastas)} FASTA files, but ALREADY_GENERATED_COUNT={ALREADY_GENERATED_COUNT}."
    )

gene_fastas = all_gene_fastas[ALREADY_GENERATED_COUNT:]

print(f"Discovered total FASTA files: {len(all_gene_fastas)}")
print(f"Selected remaining FASTA files: {len(gene_fastas)}")
if EXPECTED_REMAINING is not None and len(gene_fastas) != EXPECTED_REMAINING:
    print(f"Warning: expected {EXPECTED_REMAINING} remaining genes, got {len(gene_fastas)}")
print("First 5 selected FASTA paths:")
print(gene_fastas[:5])

In [ ]:
# -----------------------------
# VCF -> one-hot + GT matrix
# -----------------------------
BASES = ["A", "C", "G", "T"]
BASE_TO_IDX = {b: i for i, b in enumerate(BASES)}
IUPAC_TO_BASES = {
    "A": {"A"},
    "C": {"C"},
    "G": {"G"},
    "T": {"T"},
    "R": {"A", "G"},
    "Y": {"C", "T"},
    "S": {"G", "C"},
    "W": {"A", "T"},
    "K": {"G", "T"},
    "M": {"A", "C"},
    "B": {"C", "G", "T"},
    "D": {"A", "G", "T"},
    "H": {"A", "C", "T"},
    "V": {"A", "C", "G"},
    "N": {"A", "C", "G", "T"},
}

def decode_base_symbol(symbol):
    symbol = symbol.upper().strip()
    return IUPAC_TO_BASES.get(symbol, set())

def parse_gt_token(raw):
    gt = raw.split(":", 1)[0].replace("|", "/")
    if gt == ".":
        return []
    out = []
    for part in gt.split("/"):
        part = part.strip()
        if not part or part == ".":
            continue
        try:
            out.append(int(part))
        except ValueError:
            return []
    return out

def allele_index_to_bases(allele_idx, ref, alts):
    if allele_idx == 0:
        return decode_base_symbol(ref)
    i = allele_idx - 1
    if i < 0 or i >= len(alts):
        return set()
    return decode_base_symbol(alts[i])

def parse_vcf_rows(vcf_path):
    sample_ids = []
    rows = []
    with open(vcf_path, "r", encoding="utf-8") as fh:
        for line in fh:
            if line.startswith("##"):
                continue
            if line.startswith("#CHROM"):
                sample_ids = line.rstrip("\n").split("\t")[9:]
                continue
            if not line.strip():
                continue
            rows.append(line.rstrip("\n").split("\t"))
    if not sample_ids:
        raise ValueError(f"No #CHROM line in {vcf_path}")
    return sample_ids, rows

def build_onehot_from_vcf(vcf_path):
    sample_ids, rows = parse_vcf_rows(vcf_path)
    n_samples = len(sample_ids)

    kept_sites = []
    col_blocks = []
    gt_cols = []

    for parts in rows:
        pos = int(parts[1])
        ref = parts[3].upper()
        alts = [a.upper() for a in parts[4].split(",")]
        gt_calls = parts[9:]

        if len(gt_calls) != n_samples:
            continue
        if not decode_base_symbol(ref):
            continue
        if any(not decode_base_symbol(a) for a in alts):
            continue

        block = np.zeros((n_samples, 4), dtype=np.uint8)
        gt_col = np.full(n_samples, -1, dtype=np.int16)
        non_ref_seen = False

        for i, gt_raw in enumerate(gt_calls):
            allele_indices = parse_gt_token(gt_raw)
            if not allele_indices:
                continue

            gt_col[i] = int(allele_indices[0])

            base_set = set()
            for ai in allele_indices:
                base_set |= allele_index_to_bases(ai, ref, alts)

            if not base_set:
                continue

            if any(ai > 0 for ai in allele_indices):
                non_ref_seen = True

            for b in base_set:
                if b in BASE_TO_IDX:
                    block[i, BASE_TO_IDX[b]] = 1

        if not non_ref_seen:
            continue

        kept_sites.append({"pos": pos, "ref": ref, "alt": ",".join(alts)})
        col_blocks.append(block)
        gt_cols.append(gt_col)

    if not col_blocks:
        matrix = np.zeros((n_samples, 0), dtype=np.uint8)
        gt_matrix = np.zeros((n_samples, 0), dtype=np.int16)
        feature_names = []
        vcf_feature_names = []
    else:
        matrix = np.concatenate(col_blocks, axis=1)
        gt_matrix = np.column_stack(gt_cols)
        feature_names = []
        vcf_feature_names = []
        for s in kept_sites:
            for b in BASES:
                feature_names.append(f"pos{s['pos']}_{b}")
            vcf_feature_names.append(f"pos{s['pos']}")

    return {
        "sample_ids": sample_ids,
        "sites": kept_sites,
        "matrix": matrix,
        "vcf_gt_matrix": gt_matrix,
        "feature_names": feature_names,
        "vcf_feature_names": vcf_feature_names,
    }

def save_encoding_outputs(gene, result, out_root):
    gene_dir = out_root / gene
    gene_dir.mkdir(parents=True, exist_ok=True)

    sample_ids = result["sample_ids"]
    sites = result["sites"]
    matrix = result["matrix"]
    gt_matrix = result["vcf_gt_matrix"]
    feature_names = result["feature_names"]
    vcf_feature_names = result["vcf_feature_names"]

    np.save(gene_dir / "snp_onehot.npy", matrix)
    np.save(gene_dir / "vcf_gt_matrix.npy", gt_matrix)

    with open(gene_dir / "sample_ids.txt", "w", encoding="utf-8") as fh:
        fh.write("\n".join(sample_ids) + "\n")

    pd.DataFrame(sites).to_csv(gene_dir / "sites.csv", index=False)

    with open(gene_dir / "feature_names.txt", "w", encoding="utf-8") as fh:
        fh.write("\n".join(feature_names) + "\n")

    with open(gene_dir / "vcf_feature_names.txt", "w", encoding="utf-8") as fh:
        fh.write("\n".join(vcf_feature_names) + "\n")

    gt_df = pd.DataFrame(gt_matrix, columns=vcf_feature_names)
    gt_df.insert(0, "sample_id", sample_ids)
    gt_df.to_csv(gene_dir / "vcf_gt_matrix.tsv", sep="\t", index=False)

    np.savez_compressed(
        gene_dir / "snp_onehot_bundle.npz",
        matrix=matrix,
        vcf_gt_matrix=gt_matrix,
        sample_ids=np.asarray(sample_ids, dtype=object),
        positions=np.asarray([s["pos"] for s in sites], dtype=int),
        refs=np.asarray([s["ref"] for s in sites], dtype=object),
        alts=np.asarray([s["alt"] for s in sites], dtype=object),
        feature_names=np.asarray(feature_names, dtype=object),
        vcf_feature_names=np.asarray(vcf_feature_names, dtype=object),
        base_order=np.asarray(BASES, dtype=object),
    )

    return {
        "gene": gene,
        "n_samples": matrix.shape[0],
        "n_sites": len(sites),
        "n_features": matrix.shape[1],
    }

In [ ]:
# -----------------------------
# Build VCF + one-hot for selected remaining genes
# -----------------------------
vcf_rows = []
enc_rows = []

for idx, fasta_path in enumerate(gene_fastas, start=1):
    gene = fasta_path.stem
    vcf_path = VCF_DIR / f"{gene}.vcf"

    if REBUILD_VCF or (not vcf_path.exists()):
        v = write_gene_vcf(fasta_path, vcf_path)
    else:
        v = {"gene": gene, "n_strains": np.nan, "n_variant_sites": np.nan}
    vcf_rows.append(v)

    gene_dir = ENC_DIR / gene
    bundle_path = gene_dir / "snp_onehot_bundle.npz"
    if REBUILD_ENCODINGS or (not bundle_path.exists()):
        result = build_onehot_from_vcf(vcf_path)
        e = save_encoding_outputs(gene, result, ENC_DIR)
    else:
        onehot = np.load(gene_dir / "snp_onehot.npy")
        e = {
            "gene": gene,
            "n_samples": onehot.shape[0],
            "n_sites": onehot.shape[1] // 4,
            "n_features": onehot.shape[1],
        }
    enc_rows.append(e)

    if idx % 10 == 0 or idx == len(gene_fastas):
        print(f"Processed {idx}/{len(gene_fastas)} selected genes")

vcf_summary = pd.DataFrame(vcf_rows)
enc_summary = pd.DataFrame(enc_rows)
vcf_summary_path = RESULTS_DIR / "vcf_build_summary_remaining23.csv"
enc_summary_path = RESULTS_DIR / "encoding_build_summary_remaining23.csv"
vcf_summary.to_csv(vcf_summary_path, index=False)
enc_summary.to_csv(enc_summary_path, index=False)

display(enc_summary.head())
print(f"Saved VCF build summary to: {vcf_summary_path}")
print(f"Saved encoding build summary to: {enc_summary_path}")
print(f"Encodings are available under: {ENC_DIR}")